# NSSP addon 6 — E7 (koreksi bias iteratif + Monte Carlo) dan E8 (kurva power economic W)
**Sebelum Run All:** cukup tambahkan dataset panel 2020–2024. Matriks W sudah tertanam di Sel 1, dan kolom dideteksi otomatis di Sel 2 (cek tabel pemetaannya). Uji dulu dengan `QUICK = True`, lalu ubah ke `False` dan jalankan **Save Version → Save & Run All**.

Output di `/kaggle/working`: `ADD_E7a_dynamic_bc_empirical.csv`, `ADD_E7b_MC_summary.csv`, `ADD_E7b_MC_truth.csv`, `ADD_E7b_MC_raw.csv`, `ADD_E8_power_curve_econ.csv`.

In [ ]:
# Sel 1 — Konfigurasi
import os, glob
INPUT_DIR = "/kaggle/input"
OUT = "/kaggle/working"

def _find(pattern):
    hits = glob.glob(os.path.join(INPUT_DIR, "**", pattern), recursive=True)
    return hits[0] if hits else None

# Data panel: cari otomatis (isi manual jika perlu, mis. DATA_PATH = "/kaggle/input/.../file.csv")
DATA_PATH = _find("*panel*2020*2024*.csv") or _find("panel*.csv")

# Matriks W: dipakai dari dataset jika ada; jika tidak, ditulis dari salinan tertanam
# (identik dengan W_queen.csv, W_knn4.csv, W_econ.csv di nssp_addon5)
W_EMBED = {'queen': ',3301,3302,3303,3304,3305,3306,3307,3308,3309,3310,3311,3312,3313,3314,3315,3316,3317,3318,3319,3320,3321,3322,3323,3324,3325,3326,3327,3328,3329,3371,3372,3373,3374,3375,3376\n3301,0.0,0.3333333333333333,0.0,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0\n3302,0.14285714285714285,0.0,0.14285714285714285,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0\n3303,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3304,0.0,0.16666666666666666,0.16666666666666666,0.0,0.16666666666666666,0.0,0.16666666666666666,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.16666666666666666,0.16666666666666666,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3305,0.2,0.2,0.0,0.2,0.0,0.2,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3306,0.0,0.0,0.0,0.0,0.3333333333333333,0.0,0.3333333333333333,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3307,0.0,0.0,0.0,0.14285714285714285,0.14285714285714285,0.14285714285714285,0.0,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3308,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.14285714285714285,0.0,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.0,0.0,0.0,0.0,0.0\n3309,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.125,0.125,0.0,0.125,0.125,0.125,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.0,0.0,0.0\n3310,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.3333333333333333,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3311,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.2,0.0,0.2,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0\n3312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.2,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0\n3314,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.0,0.0,0.0,0.3333333333333333,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3315,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.0,0.0,0.0,0.0,0.14285714285714285,0.0,0.14285714285714285,0.0,0.14285714285714285,0.14285714285714285,0.0,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3316,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.0,0.3333333333333333,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3317,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3318,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.2,0.2,0.0,0.2,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3320,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.3333333333333333,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3321,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.2,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0\n3322,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.125,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.125,0.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.125,0.0,0.0\n3323,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3324,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0\n3325,0.0,0.0,0.0,0.2,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0\n3326,0.0,0.0,0.2,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0\n3327,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3328,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25\n3329,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25\n3371,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3372,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.0,0.3333333333333333,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3373,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3374,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.3333333333333333,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3376,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0\n', 'knn4': ',3301,3302,3303,3304,3305,3306,3307,3308,3309,3310,3311,3312,3313,3314,3315,3316,3317,3318,3319,3320,3321,3322,3323,3324,3325,3326,3327,3328,3329,3371,3372,3373,3374,3375,3376\n3301,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0\n3302,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0\n3303,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3304,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3305,0.0,0.0,0.25,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3306,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0\n3307,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0\n3308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0\n3309,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0\n3310,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0\n3311,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3314,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3315,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3316,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3317,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3318,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3320,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3321,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0\n3322,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0\n3323,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0\n3324,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0\n3325,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0\n3326,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0\n3327,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0\n3328,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25\n3329,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25\n3371,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0\n3372,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3373,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0\n3374,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0\n3375,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3376,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0\n', 'econ': ',3301,3302,3303,3304,3305,3306,3307,3308,3309,3310,3311,3312,3313,3314,3315,3316,3317,3318,3319,3320,3321,3322,3323,3324,3325,3326,3327,3328,3329,3371,3372,3373,3374,3375,3376\n3301,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.0,0.0\n3302,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3303,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3304,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3305,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3306,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3307,0.0,0.0,0.25,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3309,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3310,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3311,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3314,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3315,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3316,0.0,0.0,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3317,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3318,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3320,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3321,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3322,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3323,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3324,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3325,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3326,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3327,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25\n3328,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.25\n3329,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3371,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.25\n3372,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3373,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0\n3374,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0\n3375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.25\n3376,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.25,0.0\n'}
W_PATHS = {}
for k, txt in W_EMBED.items():
    p = _find(f"W_{k}.csv")
    if p is None:
        p = os.path.join(OUT, f"W_{k}_embedded.csv")
        with open(p, "w") as f:
            f.write(txt)
    W_PATHS[k] = p

# Nama kolom: None = deteksi otomatis di Sel 2; isi manual jika deteksi salah
COL_ID, COL_YEAR, COL_Y = None, None, None
COL_X = {"manuf": None, "lgrdp": None, "tpt": None, "sanit": None}   # urutan = naskah

YEARS = [2020, 2021, 2022, 2023, 2024]
SEED = 20260101
B_EMP = 499          # bootstrap, estimasi empiris
B_MC = 199           # bootstrap di dalam tiap replikasi Monte Carlo
MC_REPS = 200        # replikasi Monte Carlo (E7)
MAX_ITER, TOL = 25, 1e-3
POWER_REPS = 500     # E8
POWER_SCALES = [0.0, 0.25, 0.5, 0.75, 1.0]
N_JOBS = -1
QUICK = False        # True = uji cepat semua sel (~1 menit)

print("DATA_PATH:", DATA_PATH)
print("W_PATHS  :", W_PATHS)
assert DATA_PATH, "File data tidak ditemukan: isi DATA_PATH manual"

In [ ]:
# Sel 2 — Import, deteksi kolom, dan data
import numpy as np, pandas as pd, time
from scipy.optimize import minimize_scalar
from scipy.stats import chi2
from joblib import Parallel, delayed

if QUICK:
    B_EMP, B_MC, MC_REPS, POWER_REPS, MAX_ITER = 19, 9, 4, 20, 5

df = pd.read_csv(DATA_PATH)
print("Kolom data:", list(df.columns))
W_codes = set(pd.read_csv(W_PATHS["queen"], index_col=0).index.astype(str))

def _norm(v):
    try:
        return str(int(float(v)))
    except Exception:
        return str(v).strip()

def _pick(aliases, exclude=()):
    cols = [c for c in df.columns if c not in exclude]
    low = {c.lower().strip(): c for c in cols}
    for a in aliases:                      # cocok persis
        if a in low:
            return low[a]
    for a in aliases:                      # cocok sebagian
        for lc, c in low.items():
            if a in lc:
                return c
    return None

# ID: (1) kolom berisi kode BPS 3301 ... 3376, atau (2) kolom nama kab/kota -> kode BPS
BPS_JATENG = {
    "cilacap": "3301", "banyumas": "3302", "purbalingga": "3303", "banjarnegara": "3304",
    "kebumen": "3305", "purworejo": "3306", "wonosobo": "3307", "magelang": "3308",
    "boyolali": "3309", "klaten": "3310", "sukoharjo": "3311", "wonogiri": "3312",
    "karanganyar": "3313", "sragen": "3314", "grobogan": "3315", "blora": "3316",
    "rembang": "3317", "pati": "3318", "kudus": "3319", "jepara": "3320", "demak": "3321",
    "semarang": "3322", "temanggung": "3323", "kendal": "3324", "batang": "3325",
    "pekalongan": "3326", "pemalang": "3327", "tegal": "3328", "brebes": "3329",
    "kota magelang": "3371", "kota surakarta": "3372", "kota salatiga": "3373",
    "kota semarang": "3374", "kota pekalongan": "3375", "kota tegal": "3376",
}
def _name_to_code(v):
    t = str(v).lower()
    for ch in ".,_-()":
        t = t.replace(ch, " ")
    t = " ".join(t.split())
    is_kota = t.startswith("kota ") or t.endswith(" kota") or " kota " in t
    for w in ["kabupaten", "kab", "kota"]:
        t = " ".join(x for x in t.split() if x != w)
    t = t.replace("solo", "").strip()
    if t in ("surakarta", "salatiga"):
        is_kota = True
    return BPS_JATENG.get(("kota " + t) if is_kota else t)

if COL_ID is None:
    for c in df.columns:
        if set(df[c].map(_norm).unique()) >= W_codes:
            COL_ID = c; break
if COL_ID is None:
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            codes = df[c].map(_name_to_code)
            if set(codes.dropna().unique()) >= W_codes:
                bad = sorted(df.loc[codes.isna(), c].astype(str).unique())
                if bad:
                    print(f"Nama tak dikenal (baris diabaikan): {bad}")
                df["kode__"] = codes
                print(f"ID dibuat dari nama di kolom '{c}' -> kode BPS")
                COL_ID = "kode__"; break
if COL_YEAR is None:
    COL_YEAR = _pick(["tahun", "year", "thn", "periode"])
if COL_Y is None:
    COL_Y = _pick(["poverty", "kemiskinan", "persen_miskin", "pct_miskin", "p0", "miskin"])
ALIASES = {
    "manuf": ["pertumbuhanindustri", "pertumbuhan_industri", "manuf", "growth_manuf",
              "pertumbuhan", "manufaktur", "industry_growth"],
    "lgrdp": ["lgrdp", "log_grdp", "ln_grdp", "log_pdrb", "ln_pdrb", "lpdrb", "lnpdrb", "logpdrb"],
    "tpt":   ["tpt", "unemp", "pengangguran", "unemployment"],
    "sanit": ["sanit", "sanitasi", "sanitation", "jamban"],
}
for k in COL_X:
    if COL_X[k] is None:
        COL_X[k] = _pick(ALIASES[k], exclude=[COL_ID, COL_YEAR, COL_Y])
# log GRDP: jika tidak ada kolom log, ambil log dari kolom PDRB/GRDP level
if COL_X["lgrdp"] is None:
    lvl = _pick(["pdrb_adhk", "pdrb", "grdp"], exclude=[COL_ID, COL_YEAR, COL_Y])
    if lvl is not None:
        v = df[lvl].astype(float)
        if v.median() > 30:
            df["lgrdp__"] = np.log(v); print(f"log GRDP = log(kolom '{lvl}')")
        else:
            df["lgrdp__"] = v; print(f"kolom '{lvl}' sudah berskala log, dipakai apa adanya")
        COL_X["lgrdp"] = "lgrdp__"

print("\nPemetaan kolom (CEK!):")
print(f"  id={COL_ID} | tahun={COL_YEAR} | y={COL_Y}")
for k, v in COL_X.items():
    print(f"  {k:5s} -> {v}")
miss = [n for n, v in [("COL_ID", COL_ID), ("COL_YEAR", COL_YEAR), ("COL_Y", COL_Y)] if v is None] \
       + [f"COL_X['{k}']" for k, v in COL_X.items() if v is None]
assert not miss, f"Tidak terdeteksi: {miss} -> isi manual di Sel 1"

df = df[df[COL_ID].notna()].copy()
df[COL_ID] = df[COL_ID].map(_norm)
df[COL_YEAR] = df[COL_YEAR].astype(int)
df = df[df[COL_YEAR].isin(YEARS) & df[COL_ID].isin(W_codes)].sort_values([COL_YEAR, COL_ID])
XCOLS = list(COL_X.values())
ids = sorted(df[COL_ID].unique())
N, T_ALL, K = len(ids), len(YEARS), len(XCOLS)
assert len(df) == N * T_ALL, f"panel harus balanced: {len(df)} baris, N={N}, T={T_ALL}"
assert not df[[COL_Y] + XCOLS].isna().any().any(), "ada nilai kosong di y/X"
Y = df.pivot(index=COL_YEAR, columns=COL_ID, values=COL_Y).loc[YEARS, ids].values.astype(float)
X = np.stack([df.pivot(index=COL_YEAR, columns=COL_ID, values=c).loc[YEARS, ids].values.astype(float)
              for c in XCOLS], axis=2)
COL_X = list(COL_X)        # nama pendek untuk label output

def load_W(path):
    w = pd.read_csv(path, index_col=0)
    w.index, w.columns = w.index.astype(str), w.columns.astype(str)
    w = w.loc[ids, ids].values.astype(float)
    assert np.allclose(w.sum(1), 1), "W harus row-standardised"
    return w

WS = {k: load_W(p) for k, p in W_PATHS.items()}
EVS = {k: np.linalg.eigvals(w) for k, w in WS.items()}
if COL_ID == "kode__":
    _m = df[[c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c]) and c != COL_ID][0]].groupby(df[COL_ID]).first()
    print("\nNama -> kode BPS:", dict(_m))
print("\nN =", N, "T =", T_ALL, "| W:", list(WS))
print("Rata-rata per tahun (cek skala):")
print(df.groupby(COL_YEAR)[[COL_Y] + XCOLS].mean().round(3).to_string())

In [ ]:
# Sel 3 — Estimator QML spatial-lag panel (unit / two-way FE, Lee & Yu 2010a)
def lagW(W, A):
    return np.einsum("ij,tj...->ti...", W, A)

def demean(A, fe):
    A = A - A.mean(0, keepdims=True)
    if fe == "tw":
        A = A - A.mean(1, keepdims=True)
    return A.reshape(A.shape[0] * A.shape[1], -1)

def fit_lag(Yt, Z, W, ev, fe, lo=-0.99, hi=0.999):
    T, n = Yt.shape
    WY = lagW(W, Yt)
    zt = demean(Z, fe); yt = demean(Yt, fe)[:, 0]; wyt = demean(WY, fe)[:, 0]
    neff = (n - 1) * (T - 1) if fe == "tw" else n * (T - 1)
    P = np.linalg.solve(zt.T @ zt, zt.T)
    b0, b1 = P @ yt, P @ wyt
    e0, e1 = yt - zt @ b0, wyt - zt @ b1
    a, bb, c = e0 @ e0, 2 * e0 @ e1, e1 @ e1
    def nll(d):
        s2 = (a - d * bb + d * d * c) / neff
        ld = np.sum(np.log(np.abs(1 - d * ev))).real
        if fe == "tw":
            ld -= np.log(1 - d)
        return 0.5 * neff * (np.log(2 * np.pi * s2) + 1) - (T - 1) * ld
    r = minimize_scalar(nll, bounds=(lo, hi), method="bounded", options={"xatol": 1e-7})
    d = r.x
    return dict(d=d, b=b0 - d * b1, s2=(a - d * bb + d * d * c) / neff, ll=-r.fun)

def fe_given(Yt, Z, W, d, b, fe):
    R = Yt - d * lagW(W, Yt) - Z @ b
    mu = R.mean(0, keepdims=True)
    if fe == "tw":
        return mu + R.mean(1, keepdims=True) - R.mean()
    return np.repeat(mu, R.shape[0], 0)

def resid_s2(Yt, Z, W, d, b, fe):
    R = Yt - d * lagW(W, Yt) - Z @ b - fe_given(Yt, Z, W, d, b, fe)
    T, n = Yt.shape
    neff = (n - 1) * (T - 1) if fe == "tw" else n * (T - 1)
    return (R ** 2).sum() / neff

In [ ]:
# Sel 4 — SDM dinamis: desain, simulasi, spectral radius
def dyn_design(Yfull, W):
    yl = Yfull[:-1][..., None]
    WX = lagW(W, X)
    Z = np.concatenate([yl, lagW(W, yl), X[1:], WX[1:], X[:-1], WX[:-1]], axis=2)
    return Yfull[1:], Z

NAMES = (["tau", "eta"] + [f"b_{c}" for c in COL_X] + [f"th_{c}" for c in COL_X]
         + [f"bl_{c}" for c in COL_X] + [f"thl_{c}" for c in COL_X])

def fit_dyn(Yfull, W, ev, fe):
    Yt, Z = dyn_design(Yfull, W)
    r = fit_lag(Yt, Z, W, ev, fe)
    return np.r_[r["d"], r["b"]]          # [delta, tau, eta, ...]

def spec_rad(p, ev):
    d, tau, eta = p[0], p[1], p[2]
    return np.max(np.abs((tau + eta * ev) / (1 - d * ev)))

def sim_dyn(p, FE, s, W, eps):
    d, tau, eta, beta = p[0], p[1], p[2], p[3:]
    A = np.linalg.inv(np.eye(N) - d * W)
    WX = lagW(W, X)
    Ys = np.empty_like(Y); Ys[0] = Y[0]
    for t in range(1, T_ALL):
        xb = np.concatenate([X[t], WX[t], X[t-1], WX[t-1]], axis=1) @ beta
        Ys[t] = A @ (tau * Ys[t-1] + eta * (W @ Ys[t-1]) + xb + FE[t-1] + s * eps[t-1])
    return Ys

def implied(p, W, fe, Yfull=None):
    Yfull = Y if Yfull is None else Yfull
    Yt, Z = dyn_design(Yfull, W)
    FE = fe_given(Yt, Z, W, p[0], p[1:], fe)
    s = np.sqrt(resid_s2(Yt, Z, W, p[0], p[1:], fe))
    return FE, s

In [ ]:
# Sel 5 — Koreksi bias: satu langkah dan iteratif (Everaert & Pozzi 2007)
def boot_fits(p, FE, s, W, ev, fe, E):
    return np.array([fit_dyn(sim_dyn(p, FE, s, W, e), W, ev, fe) for e in E])

def bc_single(p_hat, W, ev, fe, E, Yfull=None):
    FE, s = implied(p_hat, W, fe, Yfull)
    draws = boot_fits(p_hat, FE, s, W, ev, fe, E)
    return 2 * p_hat - draws.mean(0), draws

def bc_iter(p_hat, W, ev, fe, E, Yfull=None):
    p = p_hat.copy(); hist = []
    for it in range(MAX_ITER):
        FE, s = implied(p, W, fe, Yfull)
        draws = boot_fits(p, FE, s, W, ev, fe, E)
        gap = p_hat - draws.mean(0)
        hist.append(np.max(np.abs(gap[:3])))
        if hist[-1] < TOL:
            break
        step = 1.0
        while True:
            cand = p + step * gap
            if -0.99 < cand[0] < 0.999 and spec_rad(cand, ev) < 0.995:
                break
            step *= 0.5
            if step < 1e-3:
                cand = p
                break
        p = cand
    return p, draws, it + 1, hist[-1]

def basic_interval(p_hat, p_gen, draws, ev, level=0.95):
    """Basic bootstrap. p_gen = parameter at which bootstrap samples were generated
    (single-step: p_hat; iterated: converged value)."""
    pseudo = p_hat + p_gen - draws
    a = (1 - level) / 2
    lo, hi = np.quantile(pseudo, [a, 1 - a], axis=0)
    sr = np.array([spec_rad(q, ev) for q in pseudo])
    return lo, hi, pseudo.std(0, ddof=1), sr

In [ ]:
# Sel 6 — E7a: estimasi empiris (3 W x 2 FE)
rng = np.random.default_rng(SEED)
E_emp = rng.standard_normal((B_EMP, T_ALL - 1, N))
rows, store = [], {}
for wn in WS:
    W, ev = WS[wn], EVS[wn]
    for fe in ["ind", "tw"]:
        t0 = time.time()
        p_hat = fit_dyn(Y, W, ev, fe)
        p_s, d_s = bc_single(p_hat, W, ev, fe, E_emp)
        p_i, d_i, nit, conv = bc_iter(p_hat, W, ev, fe, E_emp)
        store[(wn, fe)] = dict(p_hat=p_hat, p_single=p_s, p_iter=p_i)
        for lab, pb, pgen, dr in [("QML+BC single", p_s, p_hat, d_s), ("QML+BC iterative", p_i, p_i, d_i)]:
            lo, hi, se, sr = basic_interval(p_hat, pgen, dr, ev)
            for j, nm in enumerate(["delta", "tau", "eta"]):
                rows.append(dict(W=wn, FE=fe, method=lab, param=nm, qml=p_hat[j], est=pb[j],
                                 se=se[j], lo95=lo[j], hi95=hi[j]))
            rows.append(dict(W=wn, FE=fe, method=lab, param="spectral_radius",
                             qml=spec_rad(p_hat, ev), est=spec_rad(pb, ev),
                             se=np.std(sr, ddof=1), lo95=np.quantile(sr, .025),
                             hi95=np.quantile(sr, .975), p_stable=np.mean(sr < 1),
                             iters=nit if "iter" in lab else 1,
                             conv_gap=conv if "iter" in lab else np.nan))
        print(f"{wn:5s} {fe:3s} tau: QML {p_hat[1]:.3f} | single {p_s[1]:.3f} | "
              f"iter {p_i[1]:.3f} ({nit} it) | {time.time()-t0:.0f}s")

e7a = pd.DataFrame(rows)
e7a.to_csv(f"{OUT}/ADD_E7a_dynamic_bc_empirical.csv", index=False)

# Cek kecocokan dengan naskah (Table S27): QML tau queen unit 0.327, two-way 0.394
print("CHECK QML tau queen:", round(store[("queen", "ind")]["p_hat"][1], 3),
      round(store[("queen", "tw")]["p_hat"][1], 3), "(naskah: 0.327, 0.394)")

In [ ]:
# Sel 7 — E7b: Monte Carlo coverage (queen; nilai benar = estimasi BC iteratif)
def mc_one(r, p_true, FE, s, W, ev, fe):
    rr = np.random.default_rng([SEED, r, 1 if fe == "tw" else 0])
    Ysim = sim_dyn(p_true, FE, s, W, rr.standard_normal((T_ALL - 1, N)))
    Eb = rr.standard_normal((B_MC, T_ALL - 1, N))
    p_hat = fit_dyn(Ysim, W, ev, fe)
    p_s, d_s = bc_single(p_hat, W, ev, fe, Eb, Ysim)
    p_i, d_i, nit, _ = bc_iter(p_hat, W, ev, fe, Eb, Ysim)
    sr_true = spec_rad(p_true, ev)
    out = []
    for lab, pb, pgen, dr in [("QML", p_hat, None, None), ("QML+BC single", p_s, p_hat, d_s),
                              ("QML+BC iterative", p_i, p_i, d_i)]:
        if dr is None:
            lo = hi = np.full(3, np.nan); srlo = srhi = np.nan
        else:
            lo, hi, _, sr = basic_interval(p_hat, pgen, dr, ev)
            srlo, srhi = np.quantile(sr, [.025, .975])
        for j, nm in enumerate(["delta", "tau", "eta"]):
            out.append(dict(rep=r, FE=fe, method=lab, param=nm, true=p_true[j], est=pb[j],
                            cover=(lo[j] <= p_true[j] <= hi[j]) if dr is not None else np.nan))
        out.append(dict(rep=r, FE=fe, method=lab, param="spectral_radius", true=sr_true,
                        est=spec_rad(pb, ev),
                        cover=(srlo <= sr_true <= srhi) if dr is not None else np.nan))
    return out

W, ev = WS["queen"], EVS["queen"]
mc_rows, truth_rows = [], []
for fe in ["ind", "tw"]:
    p_true = store[("queen", fe)]["p_iter"]
    FE, s = implied(p_true, W, fe)
    truth_rows += [dict(FE=fe, param=n, true=v) for n, v in zip(["delta"] + NAMES, p_true)]
    t0 = time.time()
    res = Parallel(n_jobs=N_JOBS)(delayed(mc_one)(r, p_true, FE, s, W, ev, fe)
                                  for r in range(MC_REPS))
    mc_rows += [x for rr_ in res for x in rr_]
    print(f"MC {fe}: {time.time()-t0:.0f}s")

mc = pd.DataFrame(mc_rows)
mc.to_csv(f"{OUT}/ADD_E7b_MC_raw.csv", index=False)
pd.DataFrame(truth_rows).to_csv(f"{OUT}/ADD_E7b_MC_truth.csv", index=False)
mc["cover"] = mc["cover"].astype(float)
summ = (mc.assign(err=mc.est - mc.true)
          .groupby(["FE", "method", "param"])
          .agg(true=("true", "first"), mean=("est", "mean"), bias=("err", "mean"),
               rmse=("err", lambda e: np.sqrt(np.mean(e ** 2))),
               coverage95=("cover", "mean"), reps=("rep", "nunique"))
          .reset_index())
summ.to_csv(f"{OUT}/ADD_E7b_MC_summary.csv", index=False)
print(summ[summ.param.isin(["tau", "spectral_radius"])].to_string(index=False))

In [ ]:
# Sel 8 — E8: kurva power uji Durbin di bawah economic W (SDM statis)
def static_fit(Wt, ev, fe, Ydata, drop=None, durbin=True):
    Z = np.concatenate([X, lagW(Wt, X)], axis=2) if durbin else X
    if drop is not None:
        Z = np.delete(Z, drop, axis=2)
    return fit_lag(Ydata, Z, Wt, ev, fe)

Wq, We, evq, eve = WS["queen"], WS["econ"], EVS["queen"], EVS["econ"]
j_san = K + COL_X.index("sanit")          # kolom W*sanit dalam [X, WX]
p_rows = []
for fe in ["ind", "tw"]:
    th_q = static_fit(Wq, evq, fe, Y)["b"][K:]          # koefisien Durbin di bawah queen
    fit_e = static_fit(We, eve, fe, Y)
    beta_e, rho_e = fit_e["b"][:K], fit_e["d"]
    WXe = lagW(We, X)
    Ze = np.concatenate([X, WXe], axis=2)
    FEe = fe_given(Y, Ze, We, rho_e, fit_e["b"], fe)
    s_e = np.sqrt(fit_e["s2"])
    Ainv = np.linalg.inv(np.eye(N) - rho_e * We)
    for c in POWER_SCALES:
        rej = []
        for r in range(POWER_REPS):
            rr = np.random.default_rng([SEED, 8, r, int(100 * c), 1 if fe == "tw" else 0])
            mean = X @ beta_e + WXe @ (c * th_q) + FEe
            Ysim = np.einsum("ij,tj->ti", Ainv, mean + s_e * rr.standard_normal((T_ALL, N)))
            sdm = static_fit(We, eve, fe, Ysim)
            sar = static_fit(We, eve, fe, Ysim, durbin=False)
            r1 = static_fit(We, eve, fe, Ysim, drop=j_san)
            rej.append((2 * (sdm["ll"] - sar["ll"]) > chi2.ppf(.95, K),
                        2 * (sdm["ll"] - r1["ll"]) > chi2.ppf(.95, 1)))
        rej = np.array(rej)
        p_rows.append(dict(FE=fe, theta_scale=c, reject_joint=rej[:, 0].mean(),
                           reject_theta_sanit=rej[:, 1].mean(), reps=POWER_REPS,
                           theta_sanit_dgp=c * th_q[COL_X.index("sanit")]))
        print(p_rows[-1])

pd.DataFrame(p_rows).to_csv(f"{OUT}/ADD_E8_power_curve_econ.csv", index=False)

In [ ]:
# Sel 9 — Cek output
for f in sorted(os.listdir(OUT)):
    if f.startswith("ADD_E"):
        print(f, os.path.getsize(f"{OUT}/{f}"), "bytes")